# 🔍 Case Study: The Column Header Anomaly (Data Quality Debugging in Spark)

## 🏢 Business Scenario
Welcome back, Lead Data Engineer! RetailMax has just launched its PySpark platform migration. A junior engineer was tasked with testing the initial CSV load. They ran a test script against a subset of employee data, but reported strange behavior:
- Column names look like a specific employee's details (`1001`, `Rahul Sharma`, `Finance`, `65000`).
- Downstream filters and aggregations are functioning, but they reference these weird names.

Your mission is to review their notebook run, understand what happened, and implement the correct fix to restore schema integrity.


<a href="https://colab.research.google.com/github/himanshusar123/-Machine-Learning-Quiz-Classification-or-Regression-/blob/main/Day_7_Harbinger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Environment Setup
Before starting, we ensure PySpark is installed in the runtime environment.


In [5]:
pip install pyspark

### Step 2: Importing Spark Dependencies
We import the `SparkSession` class, which is the driver's gateway to the Spark cluster.


In [6]:
from pyspark.sql import SparkSession

### Step 3: Initializing SparkSession
We create a SparkSession app named "RetailMax". Under the hood, this sets up the JVM driver process.


In [7]:
spark = SparkSession.builder \
    .appName("RetailMax") \
    .getOrCreate()

### Step 4: Loading the Employee Dataset (The Bug Ingestion)
The engineer loads the CSV file `employees.csv` using the Spark reader option `header=True`.

> **⚠️ Red Flag:** Let's look closely at the resulting DataFrame schema in the next steps. Does the source CSV file actually have a header row?


In [8]:
df = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

### Step 5: Data Inspection (First Glimpse)
Let's call the action `.show()` to print the first 20 records to the console.

> **🔍 Audit Finding:** Notice the column names: `1001`, `Rahul Sharma`, `Finance`, `65000`. 
> The CSV file lacked a header row! By specifying `header=True`, Spark treated the very first employee record (Rahul Sharma) as the header names, resulting in an off-by-one headcount error and incorrect column labels.


In [9]:
df.show()

+----+--------------+----------+-----+
|1001|  Rahul Sharma|   Finance|65000|
+----+--------------+----------+-----+
|1002|   Priya Verma|        HR|55000|
|1003|    Amit Kumar|        IT|72000|
|1004|    Neha Gupta| Marketing|58000|
|1005|   Rohit Mehta|   Finance|68000|
|1006|  Anjali Singh|        HR|60000|
|1007|   Vikas Arora|        IT|85000|
|1008|Pooja Malhotra|     Sales|52000|
|1009|  Karan Khanna|Operations|62000|
|1010|   Simran Kaur|   Finance|70000|
|1011|  Arjun Kapoor|        IT|91000|
|1012|  Deepika Jain|        HR|57000|
|1013|Manish Agarwal| Marketing|63000|
|1014| Ritika Sharma|     Sales|54000|
|1015| Sachin Bansal|Operations|67000|
|1016|   Nitin Verma|   Finance|76000|
|1017|   Sonia Arora|        IT|89000|
|1018|   Varun Gupta|     Sales|56000|
|1019|  Megha Kapoor|        HR|61000|
|1020|Abhishek Singh| Marketing|65000|
|1021|   Kunal Mehra|   Finance|71000|
+----+--------------+----------+-----+
only showing top 20 rows


### Step 6: Count Check
Let's check the number of records. Notice that the count is 49 instead of 50 because the first row was consumed as headers.


In [10]:
df.count()

49

### Step 7: Print Schema Details
Observe the schema. Note how the column data types are inferred based on the first record's values.


In [11]:
df.printSchema()

root
 |-- 1001: integer (nullable = true)
 |-- Rahul Sharma: string (nullable = true)
 |-- Finance: string (nullable = true)
 |-- 65000: integer (nullable = true)



### Step 8: View Column Array
Checking the list of columns programmatically. The list matches the values of our first data row.


In [12]:
df.columns

['1001', 'Rahul Sharma', 'Finance', '65000']

### Step 9: Summary Statistics
Let's run `describe()` to get statistical metrics for the columns. Notice that columns like `Finance` show NULL for mean and stddev because they contain string values.


In [13]:
df.describe().show()

+-------+------------------+--------------+-------+------------------+
|summary|              1001|  Rahul Sharma|Finance|             65000|
+-------+------------------+--------------+-------+------------------+
|  count|                49|            49|     49|                49|
|   mean|            1026.0|          NULL|   NULL| 71020.40816326531|
| stddev|14.288690166235206|          NULL|   NULL|12681.170089149182|
|    min|              1002|Abhishek Singh|Finance|             52000|
|    max|              1050|   Yash Sharma|  Sales|             99000|
+-------+------------------+--------------+-------+------------------+



### Step 10: Filtering Data
Even though the column name is incorrectly labeled `Finance` (which should be `Department`), Spark can still execute filters by referencing the incorrect column identifier.


In [14]:
df.filter(
    df.Finance=="Finance"
).show()

+----+-------------+-------+-----+
|1001| Rahul Sharma|Finance|65000|
+----+-------------+-------+-----+
|1005|  Rohit Mehta|Finance|68000|
|1010|  Simran Kaur|Finance|70000|
|1016|  Nitin Verma|Finance|76000|
|1021|  Kunal Mehra|Finance|71000|
|1026| Aman Khurana|Finance|78000|
|1031|  Yash Sharma|Finance|81000|
|1037|  Ankit Gupta|Finance|79000|
|1042| Sakshi Gupta|Finance|82000|
|1047|Naveen Khanna|Finance|84000|
+----+-------------+-------+-----+



### Step 11: Column Selection
Selecting the employee name (labeled `"Rahul Sharma"`) and salary (labeled `"65000"`).


In [15]:
df.select(
"Rahul Sharma",
"65000"
).show()

+--------------+-----+
|  Rahul Sharma|65000|
+--------------+-----+
|   Priya Verma|55000|
|    Amit Kumar|72000|
|    Neha Gupta|58000|
|   Rohit Mehta|68000|
|  Anjali Singh|60000|
|   Vikas Arora|85000|
|Pooja Malhotra|52000|
|  Karan Khanna|62000|
|   Simran Kaur|70000|
|  Arjun Kapoor|91000|
|  Deepika Jain|57000|
|Manish Agarwal|63000|
| Ritika Sharma|54000|
| Sachin Bansal|67000|
|   Nitin Verma|76000|
|   Sonia Arora|89000|
|   Varun Gupta|56000|
|  Megha Kapoor|61000|
|Abhishek Singh|65000|
|   Kunal Mehra|71000|
+--------------+-----+
only showing top 20 rows


### Step 12: Duplicate Audit Runs
The following cells show repeated runs of inspections. Let's observe them.


In [16]:
df.count()

49

In [17]:
df.printSchema()

root
 |-- 1001: integer (nullable = true)
 |-- Rahul Sharma: string (nullable = true)
 |-- Finance: string (nullable = true)
 |-- 65000: integer (nullable = true)



In [19]:
df.columns

['1001', 'Rahul Sharma', 'Finance', '65000']

In [22]:
df.filter(
    df.Finance=="Finance"
).show()

+----+-------------+-------+-----+
|1001| Rahul Sharma|Finance|65000|
+----+-------------+-------+-----+
|1005|  Rohit Mehta|Finance|68000|
|1010|  Simran Kaur|Finance|70000|
|1016|  Nitin Verma|Finance|76000|
|1021|  Kunal Mehra|Finance|71000|
|1026| Aman Khurana|Finance|78000|
|1031|  Yash Sharma|Finance|81000|
|1037|  Ankit Gupta|Finance|79000|
|1042| Sakshi Gupta|Finance|82000|
|1047|Naveen Khanna|Finance|84000|
+----+-------------+-------+-----+



In [24]:
df.select(
"Rahul Sharma",
"65000"
).show()

+--------------+-----+
|  Rahul Sharma|65000|
+--------------+-----+
|   Priya Verma|55000|
|    Amit Kumar|72000|
|    Neha Gupta|58000|
|   Rohit Mehta|68000|
|  Anjali Singh|60000|
|   Vikas Arora|85000|
|Pooja Malhotra|52000|
|  Karan Khanna|62000|
|   Simran Kaur|70000|
|  Arjun Kapoor|91000|
|  Deepika Jain|57000|
|Manish Agarwal|63000|
| Ritika Sharma|54000|
| Sachin Bansal|67000|
|   Nitin Verma|76000|
|   Sonia Arora|89000|
|   Varun Gupta|56000|
|  Megha Kapoor|61000|
|Abhishek Singh|65000|
|   Kunal Mehra|71000|
+--------------+-----+
only showing top 20 rows


### Step 13: Sorting Results
Sorting the DataFrame descending by salary. Since the salary column is named `"65000"`, we must sort by that string identifier.


In [26]:
df.orderBy(
"65000",
ascending=False
).show()

+----+--------------+----------+-----+
|1001|  Rahul Sharma|   Finance|65000|
+----+--------------+----------+-----+
|1048|  Tanya Kapoor|        IT|99000|
|1034|  Sneha Kapoor|        IT|97000|
|1043| Hemant Sharma|        IT|96000|
|1022|  Shreya Nanda|        IT|94000|
|1039|   Sumit Verma|        IT|93000|
|1011|  Arjun Kapoor|        IT|91000|
|1017|   Sonia Arora|        IT|89000|
|1028|  Gaurav Saini|        IT|88000|
|1007|   Vikas Arora|        IT|85000|
|1047| Naveen Khanna|   Finance|84000|
|1042|  Sakshi Gupta|   Finance|82000|
|1031|   Yash Sharma|   Finance|81000|
|1037|   Ankit Gupta|   Finance|79000|
|1026|  Aman Khurana|   Finance|78000|
|1016|   Nitin Verma|   Finance|76000|
|1049|Prashant Singh|Operations|73000|
|1003|    Amit Kumar|        IT|72000|
|1035|  Rakesh Yadav|Operations|72000|
|1021|   Kunal Mehra|   Finance|71000|
|1041|    Ayush Goel|Operations|71000|
+----+--------------+----------+-----+
only showing top 20 rows


### Step 14: GroupBy Count Aggregation
Grouping by the department column (labeled `"Finance"`) and getting the employee count per department.


In [28]:
df.groupBy(
"Finance"
).count().show()

+----------+-----+
|   Finance|count|
+----------+-----+
|     Sales|    7|
|        HR|    9|
|   Finance|    9|
| Marketing|    7|
|        IT|   10|
|Operations|    7|
+----------+-----+



### Step 15: Average Salary Aggregation
Calculating the average salary per department by grouping by `"Finance"` and averaging the column `"65000"`.


In [30]:
from pyspark.sql.functions import avg

df.groupBy(
"Finance"
).agg(
avg("65000")
).show()

+----------+-----------------+
|   Finance|       avg(65000)|
+----------+-----------------+
|     Sales|57285.71428571428|
|        HR|          61000.0|
|   Finance|76555.55555555556|
| Marketing|64714.28571428572|
|        IT|          90400.0|
|Operations|69142.85714285714|
+----------+-----------------+



# 🛠️ Scenario Resolution: Restoring Schema Integrity

To fix the data quality issue, we must load the CSV file *without* treating the first line as a header, and either:
1. **Provide an explicit schema (Best Practice):** Ensures strict types and avoids loading overhead.
2. **Rename default columns:** Load the file with headers disabled and rename `_c0`, `_c1`... columns dynamically.

Let's write the corrected code in the following cell.


In [ ]:
# --- SOLUTION 1: Production-Grade Explicit Schema Ingestion ---
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define the exact schema matching the structure of employees.csv
explicit_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Salary", IntegerType(), True)
])

# Read CSV with headers disabled and explicit schema specified
df_correct_1 = spark.read.csv(
    "employees.csv",
    header=False,        # Important: employees.csv lacks a header row!
    schema=explicit_schema
)

print("--- Solution 1: Explicit Schema Ingestion (Verified) ---")
df_correct_1.show(5)
df_correct_1.printSchema()
print(f"Corrected Headcount: {df_correct_1.count()}")


# --- SOLUTION 2: Ingest with Auto-Generated Names, then Rename ---
df_temp = spark.read.csv(
    "employees.csv",
    header=False,        # Load all lines as records
    inferSchema=True     # Auto-detect data types
)

# Map auto-generated columns (_c0, _c1, _c2, _c3) to business names
df_correct_2 = df_temp.withColumnRenamed("_c0", "EmployeeID") \
                      .withColumnRenamed("_c1", "Name") \
                      .withColumnRenamed("_c2", "Department") \
                      .withColumnRenamed("_c3", "Salary")

print("\n--- Solution 2: Read & Rename (Verified) ---")
df_correct_2.show(5)
